## 💢 Linguistic markers of subtle discrimination in mental healthcare access

_WIP - NOT FOR DISTRIBUTION_

Triangulates human & GPT annotation decisions in $\mathcal{d}$<sub>annotated</sub> ($n$ = 1,923) to resolve discrepancies. Preprocesses training data and trains multilabel `ModernBERT` text classifier, with inverse-probability class weights, $k$-fold cross-validation, and hyperparameter tuning via grid search.  

**Pre-analysis plan filed Sep 20, 2025 on Open Science Framework: [doi: 10.17605/OSF.IO/WGU8Q](https://doi.org/10.17605/OSF.IO/WGU8Q).**

🔩 `clr_pipeline.ipynb`<br>
Simone J. Skeen x Claude Code (07-07-2026)

### 1. Prepare
Installs, imports, requisite packages; customizes outputs.
***
**Dependencies:** i.) Install via `%pip install -r requirements.txt` from project root before running; ii.) Download `!python -m spacy download en_core_web_lg --user` for spaCy `en_core_web_lg`.

In [ ]:
%%capture

%pip install -r ../../requirements.txt

In [ ]:
# Standard library
import itertools
import logging
import os
import random
import re
import sys
import time
import warnings

# Third-party
import numpy as np
import openai
import pandas as pd
import spacy
import torch

from dotenv import load_dotenv
from IPython.core.interactiveshell import InteractiveShell
from sklearn.metrics import (
    average_precision_score,
    cohen_kappa_score,
    f1_score,
    matthews_corrcoef,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

In [ ]:
# Env variables
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Output preferences
InteractiveShell.ast_node_interactivity = 'all'

pd.options.mode.copy_on_write = True
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

for category in (FutureWarning, UserWarning):
    warnings.simplefilter(action='ignore', category=category)

In [ ]:
%%script false --no-raise-error

# Project directory structure
.
└── mhp_subtle_discrimination/
    ├── config/
    ├── data/
    │   ├── raw/
    │   ├── processed/
    │   ├── qualitative/
    │   └── calibration/
    ├── annotation/
    │   ├── schema/
    │   └── triangulation/
    ├── src/
    │   └── notebooks/
    ├── models/
    │   ├── checkpoints/
    │   └── final/
    └── outputs/
        ├── logs/
        ├── figures/
        └── performance/

In [ ]:
# Set working directory to project root; define data paths
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('../..')
elif os.path.basename(os.getcwd()) == 'src':
    os.chdir('..')

# Add `src` subdirectory to import path for `annotate.py` custom modules 
sys.path.insert(0, 'src')

# Local imports 
from annotate import (
    build_prompts_per_code,
    code_texts_deductively_gpt,
    load_annotation_config,
    majority_vote_gpt,
)

# Inputs subdirectories
DATA_DIR = 'data'
DATA_RAW = f'{DATA_DIR}/raw'
DATA_PROC = f'{DATA_DIR}/processed'
DATA_QUAL = f'{DATA_DIR}/qualitative'
DATA_CALB = f'{DATA_DIR}/calibration'

ANNOTATION_DIR = 'annotation'

In [ ]:
# Import raw data
d = pd.read_excel(f'{DATA_RAW}/d_prelim.xlsx', index_col=[0])

#d.info()
#d.head(3)

### 2. GPT-Assisted Annotation Triangulation
Enables indepedent human-GPT annotation guided by a common schema via OpenAI API: human-specified per-tag prompts, .JSON &rarr; .xlsx structured outputs.
***
**Narrative summary**<br>
With Cohen's $\kappa$ scores $\lt$ 0.75 on `afrm` and `agnt` tags in $\mathcal{d}$<sub>annotated</sub>, a GPT-4o model, called via OpenAI API, served as consensus annotater, an approached informed by [Meng et al.'s (2024)](https://dl.acm.org/doi/10.1145/3778354) CHALET technique. Consistent with [Ziems et al. (2024)](https://aclanthology.org/2024.cl-1.8/), the independent GPT tagging decisions permitted a "majority vote" approach to inter-annotater agreement.

In [ ]:
# Replace blanks, whitespace, or NaN with 0
all_labels = [
    'agnt', 'afrm', 'brdn', 'fitt',
    'just', 'prbl', 'rbnd', 'refl',
    ]

d[all_labels] = (
    d[all_labels]
    .replace(r'^\s*$', np.nan, regex=True)  ### blanks/whitespace → NaN
    .fillna(0)                              ### NaN → 0
    .astype(int)                            ### ensure int type
    )

# Inspect & verify
d.head(3)

In [ ]:
#%%capture

# Loads `annotation_schema.YAML`
config = load_annotation_config()

# Specify tags to annotate
gpt_codes = [
    'afrm', 
    'agnt', 
    #'brdn',
    #'fitt', 
    #'rbnd', 
    #'refl',
    ]

# Build prompts from .YAML config
prompts_per_code = build_prompts_per_code(config, gpt_codes)

# Annotate df with GPT-4o
d = code_texts_deductively_gpt(
    d,
    prompts_per_code,
    )

# Inspect
d.head(3)

#### 2.1. "Majority Vote" triangulation scheme

In [ ]:
# Majority vote
d = majority_vote_gpt(d, gpt_codes)

# Inspect & export
d.head(3)
d.to_excel(f'{DATA_PROC}/d_prelim_gpt.xlsx', index=True)

In [ ]:
# Subset rows where GPT made a positive annotation (any *_gpt column = 1)
gpt_cols = [col for col in d.columns if col.endswith('_gpt')]

# Inspect 
#with pd.option_context('display.max_colwidth', None):
#    display(d[d[gpt_cols].eq(1).any(axis=1)])

#### 2.2. Hone train-test-ready $\mathcal{d}$<sub>annotated</sub> ($n$ = 1,923)

In [ ]:
# Drop chain-of-thought and favor `triangulate`=1 cells

# Find triangulated codes
triangulate_cols = [col for col in d.columns if col.endswith('_triangulate')]
triangulated_codes = [col.replace('_triangulate', '') for col in triangulate_cols]

# Drop interim columns for each triangulated code
cols_to_drop = []
for code in triangulated_codes:
    cols_to_drop.extend([
        code,                   ### Original human annotation
        f'{code}_gpt',          ### GPT binary decision
        f'{code}_rtnl_gpt',     ### GPT rationale
        f'{code}_expl_gpt',     ### GPT explanation
        'EmailPairID',          ### paired prospective client id
        'WithinPatientID',      ### prospective client unique id
        'FirstInPair',          ### first v. second prospective client per MHP indicator
        'MHP ID',               ### MHP unique id (PT-derived)
        'Unique ID',            ### computed unique parent study id
        'pilot',                ### pilot v. parent study indicator
        'note',                 ### human annotator notes
    ])

# Keep only columns that exist in df
cols_to_drop = [col for col in cols_to_drop if col in d.columns]
d = d.drop(columns=cols_to_drop)

# Rename *_triangulate → original name
rename_map = {f'{code}_triangulate': code for code in triangulated_codes}
d = d.rename(columns=rename_map)

# Inspect
d.info()
#d.head(3)

In [ ]:
# Export
d.to_excel(f'{DATA_PROC}/d_annotated_prelim.xlsx', index=True)

In [ ]:
################################ PRE-TRAINING-LOOP BREAKPOINT ################################

In [ ]:
# Import annotated `d_train`
d = pd.read_excel(f'{DATA_PROC}/d_annotated_prelim.xlsx', index_col = [0])

d.info()
d.head(6)

In [ ]:
################################ DIRTY APPEND FOR UNIT TESTING ################################
# Repeat df rows for rough testing purposes
# DELETE THIS CELL BEFORE PRODUCTION USE

target_rows = 1600
n_repeats = int(np.ceil(target_rows / len(d)))
d = pd.concat([d] * n_repeats, ignore_index=True)

print(f"Expanded d to {len(d)} rows ({n_repeats} repeats)")
###############################################################################################

### 2. Preprocess
Cleans, redacts, augments $\mathcal{d}$<sub>train</sub> upstream of pipeline.
***

In [ ]:
        ### TODO: SJS 2/3: standalone preprocess.py fx

# Load spaCy NER pipeline
nlp = spacy.load("en_core_web_lg")

# Redact named entities
PII_LABELS = {
    'PERSON', 'NORP', 'FAC', 'ORG', 
    'GPE', 'LOC', 'PRODUCT', 'EVENT',
    }

def redact_named_entities(doc) -> str:
    '''
    Replaces all spaCy-recognized (configurable upstream) 
    PII_LABELS with whitespace.
    '''
    chars = list(doc.text)
    for ent in sorted(doc.ents, key = lambda e: e.start_char, reverse = True):
        if ent.label_ not in PII_LABELS:
            continue
        chars[ent.start_char : ent.end_char] = list(" ")
    return "".join(chars)

texts = d['text'].astype(str).tolist()
d['text'] = [redact_named_entities(doc) for doc in nlp.pipe(texts)]

In [ ]:
# Remove stray manual redactions (pilot / Phase 1)
manual_redactions = [
    '<PII>', '<|PII|>', '[MHP NAME]', '[CITY]',   ### previous informal redaction pseudowords
    '[PATIENT NAME]', '[PHONE NUMBER]', '[URL]',
    ]

# Load prospective client names to redact from AEA-prefiled parent study PAP: RCT ID: AEARCTR-0016309
with open(f'{ANNOTATION_DIR}/prospective_client_names.txt') as f: 
    names_to_redact = [line.strip() for line in f if line.strip()]

for m in manual_redactions + names_to_redact:
    for col in ['text', 'rtnl']:
        d[col] = d[col].str.replace(
            m, 
            " ", 
            regex = False,
            )

# Remove numerals & newlines
d['text'] = d['text'].str.replace(
    r"[\d\r\n]+", 
    " ", 
    regex = True,
    )

# Replace NaN → 0 in `target` varlist
labels = [
    'afrm', 'agnt', 'brdn', 'dmnd', 'fitt', 
    'just', 'prbl', 'rbnd', 'refl',
    ]

for l in labels:
    d[l] = pd.to_numeric(d[l], errors = "coerce").fillna(0).astype(int)

#Inspect & verify
#d.info()
#d.head(6)

#### 2.1. Subset `TARGETS` (classification) vs. `THEMES` (reflexive qual)

In [ ]:
# Split labels into TARGETS (classifier) vs THEMES (qualitative analysis)
#
# Not all coded behaviors are suitable for automated classification. TARGETS contains labels
# with sufficient positive examples and clear linguistic markers for machine learning: affirming
# (afrm), agentic responses (agnt), fit discussions (fitt), and reflective listening (refl).
# THEMES captures rarer or more nuanced behaviors better suited for qualitative analysis: burden
# (brdn), demand (dmnd), justification (just), problematizing (prbl), and rebounding (rbnd).
# We subset all rows exhibiting any THEMES behavior into d_qualitative for expert reflexive
# analysis, exporting to Excel for manual review. This separation acknowledges that some
# phenomena require human interpretive depth rather than algorithmic pattern-matching.

# Labels for multilabel classification (sufficient n, clear patterns)
TARGETS = [
    'afrm', 
    'agnt', 
    'fitt', 
    'refl',
    ]

# Labels for qualitative/reflexive analysis (rare or nuanced)
THEMES = [
    'brdn', 
    'dmnd', 
    #'just', ### TODO 6/10: Delete throughout
    'prbl', 
    'rbnd',
    ]

# Print n / % descriptives
n_total = len(d)

print("TARGETS (classification)")
print("-" * 30)
for label in TARGETS:
    n = d[label].sum()
    pct = (n / n_total) * 100
    print(f"  {label}: n={n}, {pct:.1f}%")

print("\nTHEMES (reflexive qual)")
print("-" * 30)
for label in THEMES:
    n = d[label].sum()
    pct = (n / n_total) * 100
    print(f"  {label}: n={n}, {pct:.1f}%")

# Subset rows positive for any THEMES label
d_qualitative = d[d[THEMES].eq(1).any(axis=1)].copy()

# Inspect & export
print(f"\nRows with THEMES labels: {len(d_qualitative)}")
d_qualitative.head(3)
d_qualitative.to_excel(f'{DATA_QUAL}/d_qualitative.xlsx', index=True)

# Drop THEMES columns from main pipeline dataframe
d = d.drop(columns=THEMES)

In [ ]:
# Gen `trgt` indicator: dummy code pos(1) rows
d['trgt'] = d[[
    'afrm', 'agnt',
    'fitt', 'refl',
    ]].apply(lambda row: 1 if any(row) else 0, axis = 1)

def augment(df):
    '''
    Detects pos(1) `trgt` rows; duplicates as new row; replaces new row `text` 
    with expert annotator-curated `rtnl` text to augment training instance.
    '''
    new_rows = []
    for index, row in df.iterrows():
        if row['trgt'] > 0:
            new_row = row.copy()
            new_row['text'] = row['rtnl']
            new_rows.append((index + 0.5, new_row))

    if not new_rows:
        return df

    aug_df = pd.DataFrame(
        [row for _, row in new_rows],
        index = [idx for idx, _ in new_rows],
        )
    df = pd.concat([df, aug_df])
    df = df.sort_index(kind = "stable").reset_index(drop = True)
    return df

d = augment(d.copy())

# Gen `agmt` indicator to flag augmented rows
d['agmt'] = (d['text'] == d['rtnl'].shift(1)).astype(int)

# Clear `rtnl` for augmented rows
d.loc[d['agmt'] == 1, 'rtnl'] = " "

# Remove `rtnl` construct salience delineator pseudoword tokens (parent / Phase 2) / `<PII>` redactions
salience_delineators = [
    "<|AFRM|>", "<|AGNT|>", "<|BRDN|>", "<|DMND|>", "<|FITT|>",  
    "<|JUST|>", "<|PRBL|>", "<|RBND|>", "<|REFL|>",
    ]

for s in salience_delineators:
    d['text'] = d['text'].str.replace(
        s, 
        " ", 
        regex = False,
        )

# Inspect & verify

### NOTE 6/11: appropriate data structure visuallly inspected & verified 6/9

#d.shape
#d.head(6)
#d.to_excel(f'{DATA_PROC}/d_agmt_inspect.xlsx', index=True)

#### 2.3. Subset held-out $\mathcal{d}$<sub>calibrate</sub> ($n$ = 200)

In [ ]:
# Randomly sample held-out calibration set
#
# Before training, we set aside a calibration set for post-hoc threshold tuning or confidence
# calibration. This subset (n=200) is sampled randomly from the full dataset and removed from
# the training pipeline entirely. Calibration data helps adjust prediction thresholds after
# training to optimize for specific metrics (e.g., precision-recall trade-offs) without biasing
# the model. By holding this out before any preprocessing that could cause leakage, we ensure
# calibration results reflect true out-of-sample performance.

N_CALIBRATE = 200

# Sample calibration set (stratified by `trgt` if possible)
np.random.seed(56)

if len(d) > N_CALIBRATE:
    calibrate_idx = np.random.choice(len(d), size=N_CALIBRATE, replace=False)
    d_calibrate = d.iloc[calibrate_idx].copy().reset_index(drop=True)
    d = d.drop(d.index[calibrate_idx]).reset_index(drop=True)
    print(f"Calibration set: {len(d_calibrate)} rows")
    print(f"Remaining data:  {len(d)} rows")
else:
    print(f"Warning: Dataset too small ({len(d)} rows) for {N_CALIBRATE}-row calibration set. Skipping.")
    d_calibrate = pd.DataFrame()

# Export calibration set
if len(d_calibrate) > 0:
    d_calibrate.to_excel(f'{DATA_CALB}/d_calibrate.xlsx', index=True)

### 3. Prelim classifier pipeline
Multilabel ModernBERT classifier for `afrm`, `agnt`, `fitt`, `refl` targets.
***

In [ ]:
# Configuration & hyperparameters
#
# This cell establishes the central settings for our classifier. We define SEED (56) to ensure
# reproducibility—running the same code twice will yield identical results. TARGETS specifies
# which behaviors we want the model to detect: affirming client decisions (afrm), agentic/staff
# responses (agnt), fit discussions (fitt), and reflective listening (refl). MODEL_NAME points
# to ModernBERT, a state-of-the-art language model released in 2024 that outperforms older BERT
# variants. The HPARAMS dictionary centralizes tunable settings: learning_rate controls how
# quickly the model adapts, batch_size determines how many examples it processes at once, epochs
# sets training iterations, and max_len caps input text length. This structure allows easy
# experimentation without hunting through scattered code.

# Random seed
SEED = 56

# Cross-validation folds
K_FOLDS = 5

# Targets for multilabel classification (`TARGETS` vs. `THEMES` defined upstream; cf. 2.1)
TARGETS = TARGETS

# Model configuration
MODEL_NAME = "answerdotai/ModernBERT-base"

# Hyperparameter grid (RTX 4090 optimized)
HPARAMS = {
    'learning_rate': 2e-5,            # Tune: [1e-5, 2e-5, 3e-5, 5e-5]
    'batch_size': 32,                 # Tune: [16, 32, 64] - RTX 4090 handles 32 easily
    'epochs': 5,                      # Tune: [3, 5, 8, 10]
    'max_len': 512,                   # Tune: [256, 384, 512] based on text length analysis
    'weight_decay': 0.01,             # Tune: [0.0, 0.01, 0.05, 0.1]
    'warmup_ratio': 0.1,              # Tune: [0.05, 0.1, 0.15]
    'dropout': 0.1,                   # Tune: [0.1, 0.2, 0.3] - classifier head dropout
    'gradient_accumulation_steps': 1, # Tune: [1, 2, 4] - effective batch = batch_size * this
}

In [ ]:
# Seed & device setup
#
# Machine learning involves randomness—shuffling data, initializing model weights, and dropout
# regularization all use random numbers. Without controlling this randomness, results vary
# between runs, making debugging difficult and replication impossible. The set_seed() function
# "locks in" the random number generators across Python, NumPy, and PyTorch so every run
# produces identical results. Next, we detect available hardware: Apple Silicon Macs have MPS
# (Metal Performance Shaders), NVIDIA GPUs use CUDA, and everything else falls back to CPU.
# GPUs dramatically accelerate training—a task taking hours on CPU might finish in minutes on
# GPU. The device variable tells PyTorch where to run computations, and we print it so users
# can verify their hardware is being utilized properly.

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# Device selection (MPS for Apple Silicon, CUDA, or CPU)
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
    )
print(f"Device: {device}")

#### $\mathcal{d}$<sub>train</sub> / $\mathcal{d}$<sub>test</sub> 80:20 split

In [ ]:
# Train-test split (80:20)
#
# To evaluate a model fairly, we must test it on data it has never seen during training—like
# giving a student a new exam rather than the practice problems they studied. We split original
# (non-augmented) messages 80% for training and 20% for testing. "Stratified" splitting ensures
# both sets have similar proportions of positive examples, preventing scenarios where all rare
# cases end up in one set. Augmented rows (expert-curated rationale snippets duplicated from
# positive cases) are added only to training data—including them in testing would artificially
# inflate performance since they overlap with training content. We map each augmented row back
# to its "parent" original, ensuring augmented versions only appear in training if their parent
# does. This prevents data leakage, where test set information inadvertently influences training.

# Separate original vs augmented rows
d_orig = d[d['agmt'] == 0].reset_index(drop=True)
d_augm = d[d['agmt'] == 1].copy()

# Map augmented rows to parent indices
orig_positions = d.index[d['agmt'] == 0].tolist()
augm_positions = d.index[d['agmt'] == 1].tolist()
pos_to_orig_idx = {pos: i for i, pos in enumerate(orig_positions)}

parent_indices = []
for ai in augm_positions:
    parent_pos = max(op for op in orig_positions if op < ai)
    parent_indices.append(pos_to_orig_idx[parent_pos])
d_augm['_parent_idx'] = parent_indices

# Stratified split by composite target indicator
train_idx, test_idx = train_test_split(
    np.arange(len(d_orig)),
    test_size=0.2,
    random_state=SEED,
    stratify=d_orig['trgt'],
    )

d_train_orig = d_orig.iloc[train_idx].reset_index(drop=True)
d_test = d_orig.iloc[test_idx].reset_index(drop=True)

# Include augmented rows in training only (parent must be in train set)
train_set = set(train_idx)
d_augm_train = d_augm[d_augm['_parent_idx'].isin(train_set)].reset_index(drop=True)
d_train = pd.concat([d_train_orig, d_augm_train], ignore_index=True)

print(f"Train (orig + augmented): {len(d_train)}")
print(f"Test (orig only):         {len(d_test)}")

In [ ]:
# Compute class weights for TARGETS labels
#
# Class imbalance—when positive examples are rare—causes models to favor predicting the majority
# class. Inverse-frequency weighting counteracts this by penalizing errors on rare classes more
# heavily. For each TARGETS label, we compute pos_weight = n_negative / n_positive. This ratio
# tells the loss function how much more to penalize missing a positive case versus a false alarm.
# For example, if only 10% of messages are "affirming," pos_weight ≈ 9, making false negatives
# 9x costlier than false positives. These weights feed into BCEWithLogitsLoss during training,
# pushing the model to catch rare but important behaviors rather than defaulting to "negative."

# Compute pos_weight for each TARGETS label (n_neg / n_pos)
pos_weights = []
print("Class weights (pos_weight = n_neg / n_pos)")
print("-" * 40)

for label in TARGETS:
    n_pos = d_train[label].sum()
    n_neg = len(d_train) - n_pos
    if n_pos > 0:
        pw = n_neg / n_pos
    else:
        pw = 1.0  # default weight if no positive samples
    pos_weights.append(pw)
    print(f"  {label}: n_pos={n_pos}, n_neg={n_neg}, pos_weight={pw:.2f}")

# Convert to tensor for BCEWithLogitsLoss
pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float).to(device)
print(f"\npos_weight_tensor: {pos_weight_tensor}")

In [ ]:
# Multilabel Dataset class
#
# Neural networks cannot directly read text—they require numerical input. This Dataset class
# bridges that gap. The tokenizer converts each message into a sequence of token IDs (integers
# representing words or word-pieces) plus an attention mask indicating which positions contain
# real content versus padding. We pad all sequences to max_len (512 tokens) so they can be
# processed in batches efficiently. For multilabel classification, each example has multiple
# binary labels (one per target), stored as floating-point numbers for compatibility with the
# loss function. The __getitem__ method retrieves one example at a time, and PyTorch's
# DataLoader will later batch these together. This standardized interface allows the training
# loop to iterate through data without knowing the underlying text processing details.

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)

class MultilabelDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels  ### shape: (n_samples, n_labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
            )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float),
            }

In [ ]:
# Model initialization (SKIPPED - models now initialized fresh in k-fold CV loop)
#
# Here we load ModernBERT, a transformer-based language model pretrained on vast amounts of
# text. "Pretrained" means it already understands language patterns—grammar, word meanings,
# context—before seeing our specific task. We fine-tune it for our multilabel classification
# by adding a classification head with num_labels outputs (one per target). Setting
# problem_type="multi_label_classification" tells HuggingFace to use binary cross-entropy loss,
# treating each label independently—a message can exhibit multiple behaviors simultaneously
# (e.g., both affirming and reflective). The model downloads from HuggingFace Hub using our
# authentication token, then moves to the selected device (GPU/CPU). We suppress verbose
# logging to keep output clean. This pretrained foundation dramatically reduces the training
# data needed compared to training from scratch.

# logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

# model = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_NAME,
#     num_labels=len(TARGETS),
#     problem_type="multi_label_classification",
#     token=HF_TOKEN,
#     ).to(device)

In [ ]:
# Training function
#
# Training adjusts model weights to minimize prediction errors. The optimizer (AdamW) controls
# how weights update—learning_rate sets step size, weight_decay prevents overfitting by
# penalizing large weights. We use a "warmup" schedule: learning rate starts near zero, ramps
# up during early training (warmup_ratio portion), then gradually decays. This prevents
# destabilizing large updates before the model finds a good region of parameter space. Each
# epoch passes through all training data once. Gradient accumulation simulates larger batch
# sizes without extra VRAM—we accumulate gradients over multiple mini-batches before updating.
# The loss measures prediction error—lower is better. Printing epoch losses helps monitor
# progress; loss should generally decrease, though some fluctuation is normal.

def train_model(model, train_loader, hparams, device, pos_weight=None):
    optimizer = AdamW(
        model.parameters(),
        lr=hparams['learning_rate'],
        weight_decay=hparams['weight_decay'],
        )

    total_steps = len(train_loader) * hparams['epochs']
    warmup_steps = int(total_steps * hparams['warmup_ratio'])
    scheduler = get_linear_schedule_with_warmup(
        optimizer, warmup_steps, total_steps
        )

    # Custom loss with class weights if provided
    if pos_weight is not None:
        loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        loss_fn = torch.nn.BCEWithLogitsLoss()

    # Gradient accumulation steps
    accum_steps = hparams.get('gradient_accumulation_steps', 1)

    model.train()
    for epoch in range(hparams['epochs']):
        epoch_loss = 0
        optimizer.zero_grad()
        
        for i, batch in enumerate(train_loader):
            outputs = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device),
                )
            loss = loss_fn(outputs.logits, batch['labels'].to(device))
            
            # Scale loss for gradient accumulation
            loss = loss / accum_steps
            loss.backward()
            
            # Update weights after accumulating gradients
            if (i + 1) % accum_steps == 0 or (i + 1) == len(train_loader):
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
            
            epoch_loss += loss.item() * accum_steps  # Rescale for logging

        print(f"Epoch {epoch+1}/{hparams['epochs']} - Loss: {epoch_loss/len(train_loader):.4f}")

    return model

In [ ]:
# Evaluation function
#
# After training, we assess performance on held-out test data the model never saw. The model
# outputs raw scores (logits) for each label; sigmoid converts these to probabilities between
# 0 and 1. Predictions above 0.5 are classified as positive. We compute three metrics per
# label: F1-macro balances precision (accuracy of positive predictions) and recall (coverage
# of actual positives), averaging across classes to handle imbalance. AUPRC (Area Under
# Precision-Recall Curve) measures ranking quality—how well the model separates positives from
# negatives across all thresholds, crucial for rare events. MCC (Matthews Correlation
# Coefficient) provides a balanced measure even with skewed class distributions, ranging from
# -1 (total disagreement) to +1 (perfect prediction). Aggregate metrics summarize overall
# multilabel performance across all four targets combined.

def evaluate_model(model, test_loader, device, targets):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []

    with torch.no_grad():
        for batch in test_loader:
            outputs = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device),
                )
            probs = torch.sigmoid(outputs.logits)
            preds = (probs > 0.5).int()

            all_preds.append(preds.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
            all_labels.append(batch['labels'].numpy())

    all_preds = np.vstack(all_preds)
    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)

    # Per-label metrics
    results = {}
    for i, target in enumerate(targets):
        y_true = all_labels[:, i]
        y_pred = all_preds[:, i]
        y_prob = all_probs[:, i]

        results[target] = {
            'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
            'mcc': matthews_corrcoef(y_true, y_pred),
            }
        # AUPRC requires positive samples
        if y_true.sum() > 0:
            results[target]['auprc'] = average_precision_score(y_true, y_prob)
        else:
            results[target]['auprc'] = float('nan')

    # Aggregate metrics (micro-averaged across all labels)
    results['aggregate'] = {
        'f1_macro': f1_score(all_labels.ravel(), all_preds.ravel(), average='macro', zero_division=0),
        }
    if all_labels.ravel().sum() > 0:
        results['aggregate']['auprc'] = average_precision_score(all_labels.ravel(), all_probs.ravel())

    return results

In [ ]:
# Run k-fold cross-validation pipeline
#
# K-fold cross-validation provides more robust performance estimates than a single train-test
# split. We divide training data into K equal parts (folds). For each iteration, K-1 folds
# train the model while 1 fold validates it—rotating through all folds. This means every
# example serves as validation exactly once. We train a fresh model for each fold to avoid
# contamination. After all folds complete, we average metrics to get stable estimates with
# standard deviations indicating variability. Finally, we evaluate on the held-out test set
# (never seen during CV) for unbiased final performance. This approach balances thorough
# evaluation with computational cost, catching overfitting that single splits might miss.

# Prepare data arrays
train_texts = d_train['text'].values
train_labels = d_train[TARGETS].values

test_texts = d_test['text'].tolist()
test_labels = d_test[TARGETS].values

# K-fold CV on training data
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

# Store per-fold metrics
fold_results = []

print(f"Running {K_FOLDS}-fold cross-validation...")
print(f"Effective batch size: {HPARAMS['batch_size']} * {HPARAMS['gradient_accumulation_steps']} = {HPARAMS['batch_size'] * HPARAMS['gradient_accumulation_steps']}")
print("=" * 50)

# Use trgt for stratification (composite indicator)
stratify_labels = d_train['trgt'].values

# Load config once for dropout customization
model_config = AutoConfig.from_pretrained(MODEL_NAME, token=HF_TOKEN)
model_config.num_labels = len(TARGETS)
model_config.problem_type = "multi_label_classification"
model_config.classifier_dropout = HPARAMS.get('dropout', 0.1)

for fold, (train_idx, val_idx) in enumerate(skf.split(train_texts, stratify_labels)):
    print(f"\nFold {fold + 1}/{K_FOLDS}")
    print("-" * 30)
    
    # Split data for this fold
    fold_train_texts = train_texts[train_idx].tolist()
    fold_train_labels = train_labels[train_idx]
    fold_val_texts = train_texts[val_idx].tolist()
    fold_val_labels = train_labels[val_idx]
    
    # Compute fold-specific class weights
    fold_pos_weights = []
    for i, label in enumerate(TARGETS):
        n_pos = fold_train_labels[:, i].sum()
        n_neg = len(fold_train_labels) - n_pos
        pw = n_neg / n_pos if n_pos > 0 else 1.0
        fold_pos_weights.append(pw)
    fold_pos_weight_tensor = torch.tensor(fold_pos_weights, dtype=torch.float).to(device)
    
    # Create datasets and loaders (with RTX 4090 optimizations)
    fold_train_dataset = MultilabelDataset(fold_train_texts, fold_train_labels, tokenizer, HPARAMS['max_len'])
    fold_val_dataset = MultilabelDataset(fold_val_texts, fold_val_labels, tokenizer, HPARAMS['max_len'])
    
    fold_train_loader = DataLoader(
        fold_train_dataset, 
        batch_size=HPARAMS['batch_size'], 
        shuffle=True,
        pin_memory=True if device.type == 'cuda' else False,
        num_workers=4 if device.type == 'cuda' else 0,
        )
    fold_val_loader = DataLoader(
        fold_val_dataset, 
        batch_size=HPARAMS['batch_size'],
        pin_memory=True if device.type == 'cuda' else False,
        num_workers=4 if device.type == 'cuda' else 0,
        )
    
    # Fresh model for each fold
    fold_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        config=model_config,
        token=HF_TOKEN,
        ).to(device)
    
    # Train
    fold_model = train_model(fold_model, fold_train_loader, HPARAMS, device, pos_weight=fold_pos_weight_tensor)
    
    # Evaluate on validation fold
    fold_metrics = evaluate_model(fold_model, fold_val_loader, device, TARGETS)
    fold_results.append(fold_metrics)
    
    # Print fold results
    print(f"Fold {fold + 1} validation results:")
    for target, metrics in fold_metrics.items():
        if target != 'aggregate':
            print(f"  {target}: F1={metrics['f1_macro']:.4f}, MCC={metrics['mcc']:.4f}")
    
    # Clean up
    del fold_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()

# Aggregate CV results
print("\n" + "=" * 50)
print("Cross-validation summary (mean ± std)")
print("=" * 50)

cv_summary = {}
for target in TARGETS + ['aggregate']:
    cv_summary[target] = {}
    for metric in ['f1_macro', 'mcc', 'auprc']:
        values = [fold[target].get(metric, np.nan) for fold in fold_results]
        values = [v for v in values if not np.isnan(v)]
        if values:
            cv_summary[target][metric] = {'mean': np.mean(values), 'std': np.std(values)}

for target, metrics in cv_summary.items():
    print(f"\n{target}:")
    for metric, stats in metrics.items():
        print(f"  {metric}: {stats['mean']:.4f} ± {stats['std']:.4f}")

# Final evaluation on held-out test set
print("\n" + "=" * 50)
print("Final test set evaluation")
print("=" * 50)

# Train final model on all training data
final_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=model_config,
    token=HF_TOKEN,
    ).to(device)

train_dataset = MultilabelDataset(train_texts.tolist(), train_labels, tokenizer, HPARAMS['max_len'])
test_dataset = MultilabelDataset(test_texts, test_labels, tokenizer, HPARAMS['max_len'])

train_loader = DataLoader(
    train_dataset, 
    batch_size=HPARAMS['batch_size'], 
    shuffle=True,
    pin_memory=True if device.type == 'cuda' else False,
    num_workers=4 if device.type == 'cuda' else 0,
    )
test_loader = DataLoader(
    test_dataset, 
    batch_size=HPARAMS['batch_size'],
    pin_memory=True if device.type == 'cuda' else False,
    num_workers=4 if device.type == 'cuda' else 0,
    )

print("\nTraining final model on full training set...")
final_model = train_model(final_model, train_loader, HPARAMS, device, pos_weight=pos_weight_tensor)

# Evaluate on test set
test_results = evaluate_model(final_model, test_loader, device, TARGETS)

print("\nTest set results:")
for target, metrics in test_results.items():
    print(f"\n{target}:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")

#### 3.1. Grid search (optional)

In [ ]:
# Hyperparameter grid search
#
# Grid search systematically explores hyperparameter combinations to find optimal settings.
# We define a search space for the most impactful parameters (learning rate, batch size, epochs,
# weight decay, dropout) and train models for each combination using k-fold CV. Results are
# stored in a DataFrame sorted by mean F1 score. A phased approach is recommended: Phase 1
# sweeps learning_rate × batch_size (highest impact), Phase 2 refines secondary parameters
# with best Phase 1 settings. Set RUN_GRID_SEARCH = True to execute; this is computationally
# expensive (hours on GPU). Results export to CSV for analysis. The best configuration can
# then be copied back to HPARAMS for final training.

# Toggle grid search execution
RUN_GRID_SEARCH = False  # Set to True to run (expensive!)

if RUN_GRID_SEARCH:
    # Define search grid (Phase 1: most impactful params)
    GRID = {
        'learning_rate': [1e-5, 2e-5, 5e-5],
        'batch_size': [16, 32],
    }
    
    # Fixed params during this search phase
    FIXED = {
        'epochs': 5,
        'max_len': 512,
        'weight_decay': 0.01,
        'warmup_ratio': 0.1,
        'dropout': 0.1,
        'gradient_accumulation_steps': 1,
    }
    
    # Generate all combinations
    keys = list(GRID.keys())
    combinations = list(itertools.product(*GRID.values()))
    
    print(f"Grid search: {len(combinations)} combinations * {K_FOLDS} folds")
    print("=" * 60)
    
    # Store results
    grid_results = []
    
    for combo_idx, combo in enumerate(tqdm(combinations, desc="Grid Search")):
        hparams = dict(zip(keys, combo))
        hparams.update(FIXED)
        
        print(f"\n[{combo_idx + 1}/{len(combinations)}] {hparams}")
        
        # Load config with current dropout (set num_labels and problem_type on config)
        grid_config = AutoConfig.from_pretrained(MODEL_NAME, token=HF_TOKEN)
        grid_config.num_labels = len(TARGETS)
        grid_config.problem_type = "multi_label_classification"
        grid_config.classifier_dropout = hparams.get('dropout', 0.1)
        
        # K-fold CV for this combination
        combo_fold_results = []
        
        for fold, (tr_idx, val_idx) in enumerate(skf.split(train_texts, stratify_labels)):
            # Split data
            fold_tr_texts = train_texts[tr_idx].tolist()
            fold_tr_labels = train_labels[tr_idx]
            fold_val_texts = train_texts[val_idx].tolist()
            fold_val_labels = train_labels[val_idx]
            
            # Compute class weights
            fold_pw = []
            for i in range(len(TARGETS)):
                n_pos = fold_tr_labels[:, i].sum()
                n_neg = len(fold_tr_labels) - n_pos
                fold_pw.append(n_neg / n_pos if n_pos > 0 else 1.0)
            fold_pw_tensor = torch.tensor(fold_pw, dtype=torch.float).to(device)
            
            # Create datasets
            tr_dataset = MultilabelDataset(fold_tr_texts, fold_tr_labels, tokenizer, hparams['max_len'])
            val_dataset = MultilabelDataset(fold_val_texts, fold_val_labels, tokenizer, hparams['max_len'])
            
            tr_loader = DataLoader(
                tr_dataset, 
                batch_size=hparams['batch_size'], 
                shuffle=True,
                pin_memory=True if device.type == 'cuda' else False,
                num_workers=4 if device.type == 'cuda' else 0,
            )
            val_loader = DataLoader(
                val_dataset, 
                batch_size=hparams['batch_size'],
                pin_memory=True if device.type == 'cuda' else False,
                num_workers=4 if device.type == 'cuda' else 0,
            )
            
            # Train model
            grid_model = AutoModelForSequenceClassification.from_pretrained(
                MODEL_NAME,
                config=grid_config,
                token=HF_TOKEN,
            ).to(device)
            
            grid_model = train_model(grid_model, tr_loader, hparams, device, pos_weight=fold_pw_tensor)
            
            # Evaluate
            fold_metrics = evaluate_model(grid_model, val_loader, device, TARGETS)
            combo_fold_results.append(fold_metrics)
            
            # Cleanup
            del grid_model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            elif torch.backends.mps.is_available():
                torch.mps.empty_cache()
        
        # Aggregate fold metrics for this combination
        f1_scores = [f['aggregate']['f1_macro'] for f in combo_fold_results]
        auprc_scores = [f['aggregate'].get('auprc', np.nan) for f in combo_fold_results]
        auprc_scores = [v for v in auprc_scores if not np.isnan(v)]
        
        result = {
            **hparams,
            'mean_f1': np.mean(f1_scores),
            'std_f1': np.std(f1_scores),
            'mean_auprc': np.mean(auprc_scores) if auprc_scores else np.nan,
            'std_auprc': np.std(auprc_scores) if auprc_scores else np.nan,
        }
        grid_results.append(result)
        
        print(f"  → F1: {result['mean_f1']:.4f} ± {result['std_f1']:.4f}")
    
    # Convert to DataFrame and sort
    grid_df = pd.DataFrame(grid_results).sort_values('mean_f1', ascending=False)
    
    print("\n" + "=" * 60)
    print("Grid Search Results (sorted by mean F1)")
    print("=" * 60)
    print(grid_df.to_string(index=False))
    
    # Export results
    grid_df.to_csv(f'{DATA_DIR}/grid_search_results.csv', index=False)
    print(f"\nResults saved to {DATA_DIR}/grid_search_results.csv")
    
    # Best configuration
    best = grid_df.iloc[0]
    print(f"\nBest configuration:")
    for param in keys:
        print(f"  {param}: {best[param]}")
    print(f"  mean_f1: {best['mean_f1']:.4f} ± {best['std_f1']:.4f}")
else:
    print("Grid search disabled. Set RUN_GRID_SEARCH = True to execute.")